## GPCRmd-NMR aligns with experimental NMR observations - Neurotensin Receptor

Here we compare the CS predictor correlations for the 3 different predictors (SPARTA+, SHIFTX2 and UCBShift2.0) for the Neurotensin Receptor (NTSR1) with the experimental NMR data.


In [6]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import statistics

### Define functions to compute correlations

In [7]:
def read_experimental_cs(file_path, shift=0, resid_column=19, cs_column=10, atom_column=7, resname_column=6):
    data_dict = {}
    with open(file_path, 'r') as file:
        for line in file:
            columns = line.strip().split()
            if len(columns) <= max(resid_column, cs_column, atom_column, resname_column):
                continue
            resname = columns[resname_column]
            resid = int(columns[resid_column]) + shift
            atomname = columns[atom_column]
            chemshift = float(columns[cs_column])
            atomname_base = ''.join(filter(str.isalpha, atomname))
            data_dict[f"{resid}_{resname}_{atomname_base}"] = chemshift
    return data_dict

def read_predicted_csv(file_path):
    pred_dict = {}
    with open(file_path, 'r') as file:
        lines = file.readlines()[1:]  # skip header
        for line in lines:
            parts = line.strip().split(";")
            if len(parts) < 6:
                continue
            resid = parts[0]
            atom = parts[1]
            resname = parts[2]
            try:
                cs_values = [float(x) for x in parts[4:-1]]
            except ValueError:
                continue
            if not cs_values:
                continue
            atomname_base = ''.join(filter(str.isalpha, atom))
            key = f"{resid}_{resname}_{atomname_base}"
            pred_dict[key] = round(statistics.mean(cs_values), 4)
    return pred_dict

def compute_correlation(exp_dict, pred_dict, atom_type):
    exp_vals, pred_vals = [], []
    for k, exp_val in exp_dict.items():
        if k in pred_dict and k.endswith(f"_{atom_type}"):
            exp_vals.append(exp_val)
            pred_vals.append(pred_dict[k])
    if len(exp_vals) < 2:
        return None
    exp_arr, pred_arr = np.array(exp_vals), np.array(pred_vals)
    r, _ = pearsonr(exp_arr, pred_arr)
    rmse = np.sqrt(np.mean((exp_arr - pred_arr) ** 2))
    return r, rmse, len(exp_vals), exp_arr, pred_arr

def plot_correlation(exp_arr, pred_arr, r, rmse, atom_type, predictor, output_path, replicate_id):
    plt.figure(figsize=(4, 4))
    # swap axes: x = predicted, y = experimental
    plt.scatter(pred_arr, exp_arr, alpha=0.6, color='black')
    mn = min(np.min(exp_arr), np.min(pred_arr))
    mx = max(np.max(exp_arr), np.max(pred_arr))
    plt.plot([mn, mx], [mn, mx], 'r--', label="y=x")
    plt.xlabel('Predicted CS (ppm)', size=12)
    plt.ylabel('Experimental CS (ppm)', size=12)
    # invert x and y axis
    plt.gca().invert_xaxis()
    plt.gca().invert_yaxis()
    plt.text(0.05, 0.95, f"r = {r:.3f}\nRMSE = {rmse:.3f}",
             transform=plt.gca().transAxes, fontsize=10, va='top',
             bbox=dict(facecolor='white', alpha=0.8))
    plt.title(f"Replicate {replicate_id}", size=16)
    plt.tight_layout()
    plt.grid(True)
    plt.savefig(output_path, dpi=200)
    plt.close()

def plot_multi_replicate_figure(plot_data, atom_type, predictor, output_dir):
    """Create a 3x4 grid of replicate scatter plots (10 plots, 2 empty) with x=predicted, y=experimental."""
    fig, axes = plt.subplots(4, 3, figsize=(12, 16))
    axes = axes.flatten()

    for i in range(12):
        ax = axes[i]
        if i < len(plot_data):
            exp_arr, pred_arr, r, rmse, rep_id = plot_data[i]
            # swap axes: x = predicted, y = experimental
            ax.scatter(pred_arr, exp_arr, alpha=0.6, color='black')
            mn = min(np.min(exp_arr), np.min(pred_arr))
            mx = max(np.max(exp_arr), np.max(pred_arr))
            ax.plot([mn, mx], [mn, mx], 'r--')
            ax.invert_xaxis()
            ax.invert_yaxis()
            ax.text(0.05, 0.95, f"r={r:.3f}\nRMSE={rmse:.3f}", transform=ax.transAxes,
                    fontsize=10, va='top', bbox=dict(facecolor='white', alpha=0.8))
            ax.set_title(f"Replicate {rep_id}", fontsize=12)
            ax.set_xlabel("Predicted CS (ppm)", fontsize=12)
            ax.set_ylabel("Experimental CS (ppm)", fontsize=12)
            ax.tick_params(labelsize=12)
            ax.grid(True)
        else:
            ax.axis('off')  # leave last two blank

    plt.suptitle(f"{predictor} — {atom_type}: 10 Replicates", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    output_path = os.path.join(output_dir, f"{predictor}_{atom_type}_all_replicates.png")
    plt.savefig(output_path, dpi=300)
    plt.close()

### Define initial settings and parameters

In [8]:
# === SETTINGS ===
predictors = ["ShiftX2", "UCBShift", "SPARTA"]
replicates = [f"rep{i}.csv" for i in range(1, 11)]
atom_types = ["CA", "CB", "C", "N", "H"]
base_dir = "data"
output_dir = "results_predictors"
os.makedirs(output_dir, exist_ok=True)

### Compute correlations for each predictor and replicate, and save results

In [ ]:
experimental_path = os.path.join(base_dir, "cs_data.txt")
exp_dict = read_experimental_cs(experimental_path, 0, 20, 11, 8, 7)

summary = []
all_results = []  # will store replicate-level correlations for per-atom tables

for predictor in predictors:
    print(f"\nProcessing predictor: {predictor}")
    pred_dir = os.path.join(base_dir, predictor)
    
    replicate_results = []
    
    for atom in atom_types:
        # Create output directory for this predictor/atom
        atom_out_dir = os.path.join(output_dir, predictor, atom)
        os.makedirs(atom_out_dir, exist_ok=True)

        # collect replicate plots for multi-panel figure
        plot_data = []

        for rep in replicates:
            rep_path = os.path.join(pred_dir, rep)
            pred_dict = read_predicted_csv(rep_path)
            result = compute_correlation(exp_dict, pred_dict, atom)
            if result:
                r, rmse, n, exp_arr, pred_arr = result
                replicate_id = int(rep.replace("rep", "").replace(".csv", ""))
                replicate_results.append({
                    "Predictor": predictor,
                    "Replicate": rep,
                    "AtomType": atom,
                    "Pearson_r": round(r, 4),
                    "RMSE": round(rmse, 4),
                    "N_points": n
                })
                # save for atom-specific correlation table
                all_results.append({
                    "AtomType": atom,
                    "Predictor": predictor,
                    "Replicate": rep,
                    "Pearson_r": round(r, 3)
                })

                plot_path = os.path.join(atom_out_dir, f"{rep.replace('.csv', '')}_correlation.png")
                plot_correlation(exp_arr, pred_arr, r, rmse, atom, predictor, plot_path, replicate_id)
                plot_data.append((exp_arr, pred_arr, r, rmse, replicate_id))

        # create the multi-plot figure for this atom
        if plot_data:
            plot_multi_replicate_figure(plot_data, atom, predictor, atom_out_dir)

    # Combine replicate results for summary
    df_rep = pd.DataFrame(replicate_results)
    for atom in atom_types:
        sub = df_rep[df_rep["AtomType"] == atom]
        if not sub.empty:
            mean_r = sub["Pearson_r"].mean()
            std_r = sub["Pearson_r"].std()
            summary.append({
                "Predictor": predictor,
                "AtomType": atom,
                "Mean_r": round(mean_r, 4),
                "SD_r": round(std_r, 4),
                "N_replicates": len(sub)
            })



Processing predictor: ShiftX2
